In [11]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,departure_time,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class,delay_minutes
0,4227,02:58,788.12,103.94,2,sunny,Fri,0,3,61.463731,intermediate,11
1,4676,05:19,408.42,96.60,7,sunny,Sat,0,13,125.058439,premium,17
2,800,18:44,440.24,92.54,1,sunny,Sun,0,12,178.797255,standard,0
3,3671,06:27,345.01,104.57,5,sunny,Sat,0,11,137.304807,standard,5
4,4193,22:09,729.77,82.12,7,sunny,Wed,0,6,193.314124,premium,27


In [12]:
#preprocessing - ordinal encoding by hand, label encoding and scaling

df_train.weather = df_train.weather.map({"sunny": 0, "rain": 1, "snow": 2, "fog": 3})
df_train.comfort_class = df_train.comfort_class.map({"standard": 0, "intermediate": 1, "premium": 2})

df_test.weather = df_test.weather.map({"sunny": 0, "rain": 1, "snow": 2, "fog": 3})
df_test.comfort_class = df_test.comfort_class.map({"standard": 0, "intermediate": 1, "premium": 2})

from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
df_train.weekday = enc.fit_transform(df_train.weekday)
df_test.weekday = enc.transform(df_test.weekday)

df_train.head()

,SampleID,departure_time,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class,delay_minutes
0,4227,02:58,788.12,103.94,2,0,0,0,3,61.463731,1,11
1,4676,05:19,408.42,96.60,7,0,2,0,13,125.058439,2,17
2,800,18:44,440.24,92.54,1,0,3,0,12,178.797255,0,0
3,3671,06:27,345.01,104.57,5,0,2,0,11,137.304807,0,5
4,4193,22:09,729.77,82.12,7,0,6,0,6,193.314124,2,27


In [14]:
df_train.columns

Index(['SampleID', 'departure_time', 'distance_km', 'avg_speed_kmh',
       'num_stops', 'weather', 'weekday', 'special_events', 'num_cars',
       'ticket_price', 'comfort_class', 'delay_minutes'],
      dtype='object')

In [16]:
features = ['distance_km', 'avg_speed_kmh', 'num_stops', 'weather', 'weekday', 'special_events', 'num_cars', 'ticket_price', 'comfort_class']
target = 'delay_minutes'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class
0,1.719000,1.034528,-0.841728,-0.719037,-1.481059,-0.322107,-1.625539,-0.941430,0.401239
1,-0.057173,0.721845,0.905502,-0.719037,-0.484887,-0.322107,1.293106,0.282567,1.691396
2,0.091676,0.548889,-1.191174,-0.719037,0.013199,-0.322107,1.001241,1.316870,-0.888918
3,-0.353794,1.061366,0.206610,-0.719037,-0.484887,-0.322107,0.709377,0.518271,-0.888918
4,1.446049,0.104997,0.905502,-0.719037,1.507458,-0.322107,-0.749946,1.596274,1.691396


In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

model = LinearRegression()

cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')

array([-4.32873897, -4.30297845, -4.35770219, -4.3064016 , -4.12547858])

In [22]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [23]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "delay_minutes": y_preds
})

df_preds

,SampleID,delay_minutes
0,1501,28.484448
1,2586,5.391610
2,2653,-0.619568
3,1055,13.506249
4,705,5.784394
...,...,...
995,4711,19.850069
996,2313,18.331552
997,3214,16.288844
998,2732,11.553074


In [24]:
df_preds.to_csv("submission.csv", index=False)